# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 42 • Parameter-Efficient Fine-Tuning with PEFT and LoRA

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate to Advanced  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU by default

---

## Scope

This lesson introduces parameter-efficient fine-tuning (PEFT), with emphasis
on low-rank adaptation (LoRA). It explains why adapters reduce trainable
parameters, how low-rank update matrices are inserted into Transformer
projections, how adapter checkpoints are saved and merged, and how PEFT fits
into Hugging Face workflows.

The notebook contains:

1. a complete offline CPU experiment that implements LoRA from first
   principles and compares it with full fine-tuning;
2. optional Hugging Face `peft`, Transformers, and quantization workflow
   templates that remain disabled by default.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain parameter-efficient fine-tuning;
- distinguish full fine-tuning, feature extraction, adapters, and LoRA;
- derive the LoRA update \(W + BA\);
- interpret rank, scaling, and LoRA dropout;
- identify suitable target modules;
- freeze base-model parameters while training adapters;
- count trainable and total parameters;
- compare full fine-tuning and LoRA under identical data splits;
- save adapter-only checkpoints;
- merge LoRA weights into base weights;
- explain QLoRA and k-bit base-model loading;
- structure Hugging Face PEFT workflows;
- document reproducibility, licensing, and deployment constraints.

## Table of Contents

1. Why Parameter-Efficient Fine-Tuning
2. Full Fine-Tuning Versus PEFT
3. Adapter Families
4. LoRA Mathematics
5. Rank and Expressiveness
6. Scaling and Dropout
7. Target Modules
8. Adapter Checkpoints
9. Merging Adapters
10. QLoRA Foundations
11. Offline Domain-Adaptation Dataset
12. Data Splits
13. Vocabulary and Tokenization
14. Dataset and Dynamic Padding
15. Self-Attention Block
16. Tiny Transformer Classifier
17. Base-Model Pretraining
18. LoRA Linear Layer
19. Adapter Injection
20. Parameter Counting
21. Full Fine-Tuning Baseline
22. LoRA Fine-Tuning
23. Learning Curves
24. Validation Comparison
25. Test Evaluation
26. Per-Class Metrics
27. Confusion Matrices
28. Confidence and Error Analysis
29. Adapter-Only State
30. Saving and Reloading an Adapter
31. Merging LoRA Weights
32. Merge Equivalence Check
33. Rank Comparison
34. Memory and Storage Accounting
35. Optional Hugging Face Setup
36. Optional PEFT LoRA Configuration
37. Optional Training Structure
38. Optional Adapter Saving and Loading
39. Optional Multiple Adapters
40. Optional QLoRA Structure
41. Target-Module Inspection
42. Failure Modes
43. Arabic and Multilingual Considerations
44. Reproducibility and Reporting
45. Knowledge Check
46. Exercises
47. Summary and Next Lesson

# 1. Why Parameter-Efficient Fine-Tuning

Modern pretrained models can contain millions or billions of parameters.
Updating and storing a full model copy for every task may be expensive.

PEFT methods keep most pretrained weights frozen and optimize a much smaller
set of task-specific parameters.

In [ ]:
import copy
import importlib.util
import math
import platform
import random
import re
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

motivations = pd.DataFrame(
    [
        ("Training memory", "freeze most base parameters"),
        ("Checkpoint storage", "save adapter weights only"),
        ("Multi-task deployment", "reuse one shared base model"),
        ("Experiment speed", "optimize fewer parameters"),
    ],
    columns=["Constraint", "PEFT response"],
)

motivations


# 2. Full Fine-Tuning Versus PEFT

In [ ]:
strategy_table = pd.DataFrame(
    [
        (
            "Feature extraction",
            "task head only",
            "lowest",
            "limited adaptation",
        ),
        (
            "LoRA",
            "low-rank adapters and head",
            "low",
            "target-module sensitivity",
        ),
        (
            "Full fine-tuning",
            "all parameters",
            "highest",
            "memory and storage cost",
        ),
    ],
    columns=[
        "Strategy",
        "Updated parameters",
        "Relative cost",
        "Main limitation",
    ],
)

strategy_table

# 3. Adapter Families

PEFT includes several families:

- low-rank updates;
- bottleneck adapters;
- prompt tuning;
- prefix tuning;
- soft prompts;
- selective parameter tuning.

LoRA is widely used because it can be inserted into existing linear
projections and merged into the base weights.

# 4. LoRA Mathematics

For a frozen weight matrix:

\[
W \in \mathbb{R}^{d_{out} \times d_{in}}
\]

LoRA learns:

\[
A \in \mathbb{R}^{r \times d_{in}},
\qquad
B \in \mathbb{R}^{d_{out} \times r}
\]

The adapted transformation is:

\[
y = xW^T + \frac{\alpha}{r}xA^TB^T
\]

where \(r\) is the rank and \(lpha\) is the scaling parameter.

In [ ]:
def lora_parameter_count(
    input_features: int,
    output_features: int,
    rank: int,
) -> int:
    return (
        rank * input_features
        + output_features * rank
    )


example_counts = pd.DataFrame(
    [
        (
            rank,
            lora_parameter_count(
                768,
                768,
                rank,
            ),
            768 * 768,
        )
        for rank in [2, 4, 8, 16]
    ],
    columns=[
        "Rank",
        "LoRA parameters",
        "Original matrix parameters",
    ],
)

example_counts[
    "LoRA percentage"
] = (
    100.0
    * example_counts[
        "LoRA parameters"
    ]
    / example_counts[
        "Original matrix parameters"
    ]
)

example_counts

# 5. Rank and Expressiveness

Lower rank reduces trainable parameters but limits the update subspace.
Higher rank increases capacity and memory use.

Rank should be selected with validation data rather than assumed to be
universally optimal.

# 6. Scaling and Dropout

The common scaling factor is:

\[
\frac{\alpha}{r}
\]

LoRA dropout regularizes the adapter path without changing the frozen base
path.

# 7. Target Modules

LoRA is commonly applied to attention projections such as:

- query projection;
- value projection;
- key projection;
- output projection;
- feed-forward projections.

Module names vary by architecture. They must be inspected before configuration.

In [ ]:
target_tradeoffs = pd.DataFrame(
    [
        ("Query and value", "small common baseline"),
        ("All attention projections", "more adaptation capacity"),
        ("Attention and feed-forward", "higher capacity and cost"),
    ],
    columns=["Target set", "Trade-off"],
)

target_tradeoffs

# 8. Adapter Checkpoints

Adapter checkpoints normally contain:

- adapter configuration;
- adapter weights;
- task-head weights when configured;
- metadata identifying the base checkpoint.

They do not usually contain a complete copy of the base model.

# 9. Merging Adapters

LoRA can be merged into a base matrix:

\[
W_{merged} = W + \frac{\alpha}{r}BA
\]

Merging removes adapter computation during inference, but the merged model is
no longer a small adapter-only artifact.

# 10. QLoRA Foundations

QLoRA combines:

- a quantized frozen base model;
- trainable LoRA adapters;
- higher-precision adapter computation.

Quantization reduces base-model memory, while LoRA limits the number of
trainable parameters.

# 11. Offline Domain-Adaptation Dataset

The experiment first trains a base classifier on simple domain statements.
It then adapts that model to more varied user-style requests.

In [ ]:
base_records = [
    ("doctor treats patient", "health"),
    ("nurse gives medicine", "health"),
    ("hospital reviews diagnosis", "health"),
    ("clinic provides treatment", "health"),
    ("exercise improves health", "health"),
    ("nutrition supports recovery", "health"),
    ("patient visits doctor", "health"),
    ("medical team checks report", "health"),

    ("bank approves loan", "finance"),
    ("customer requests refund", "finance"),
    ("payment failed on card", "finance"),
    ("invoice shows charge", "finance"),
    ("account receives transfer", "finance"),
    ("bank reviews transaction", "finance"),
    ("loan needs payment", "finance"),
    ("billing service updates fee", "finance"),

    ("software update failed", "technology"),
    ("server reports error", "technology"),
    ("network connection stopped", "technology"),
    ("application needs installation", "technology"),
    ("computer stores data", "technology"),
    ("device cannot connect", "technology"),
    ("upload interrupted by network", "technology"),
    ("system restarts server", "technology"),

    ("flight reaches airport", "travel"),
    ("tourist books hotel", "travel"),
    ("passenger collects luggage", "travel"),
    ("ticket reports delay", "travel"),
    ("airport changes gate", "travel"),
    ("hotel confirms reservation", "travel"),
    ("journey starts tomorrow", "travel"),
    ("tourist visits museum", "travel"),
]

adaptation_records = [
    ("where can I find a doctor for this problem", "health"),
    ("the patient needs help with recovery", "health"),
    ("please explain the medical diagnosis", "health"),
    ("the clinic changed my treatment plan", "health"),
    ("medicine caused a new health concern", "health"),
    ("I need information about hospital care", "health"),
    ("the nurse scheduled another examination", "health"),
    ("nutrition may improve the patient's condition", "health"),

    ("why was this amount charged to my card", "finance"),
    ("the bank has not processed my refund", "finance"),
    ("I need to change the loan payment date", "finance"),
    ("my account does not show the transfer", "finance"),
    ("the invoice contains an incorrect fee", "finance"),
    ("please review this billing transaction", "finance"),
    ("the payment was rejected again", "finance"),
    ("the customer wants a card refund", "finance"),

    ("the application cannot reach the remote server", "technology"),
    ("a network error interrupted the upload", "technology"),
    ("the latest software installation keeps failing", "technology"),
    ("my computer restarts after the system update", "technology"),
    ("the device cannot connect to the service", "technology"),
    ("important data disappeared from the server", "technology"),
    ("the program displays a technical error", "technology"),
    ("the network service became unavailable", "technology"),

    ("the airline changed my departure gate", "travel"),
    ("my luggage did not arrive at the airport", "travel"),
    ("the hotel cannot find my reservation", "travel"),
    ("the passenger needs a new travel ticket", "travel"),
    ("our flight was delayed until tomorrow", "travel"),
    ("the tourist wants directions to the museum", "travel"),
    ("the journey includes a stay in Boston", "travel"),
    ("the airport updated the arrival time", "travel"),
]

base_frame = pd.DataFrame(
    base_records,
    columns=["text", "label"],
)

adaptation_frame = pd.DataFrame(
    adaptation_records,
    columns=["text", "label"],
)

base_frame["label"].value_counts()

# 12. Data Splits

In [ ]:
adaptation_train, adaptation_test = train_test_split(
    adaptation_frame,
    test_size=0.25,
    random_state=42,
    stratify=adaptation_frame["label"],
)

adaptation_train, adaptation_validation = train_test_split(
    adaptation_train,
    test_size=1 / 3,
    random_state=42,
    stratify=adaptation_train["label"],
)

adaptation_train = adaptation_train.reset_index(
    drop=True
)
adaptation_validation = adaptation_validation.reset_index(
    drop=True
)
adaptation_test = adaptation_test.reset_index(
    drop=True
)

pd.Series(
    {
        "base pretraining": len(base_frame),
        "adaptation training": len(adaptation_train),
        "adaptation validation": len(adaptation_validation),
        "adaptation test": len(adaptation_test),
    }
)

# 13. Vocabulary and Tokenization

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


all_training_texts = pd.concat(
    [
        base_frame["text"],
        adaptation_train["text"],
    ],
    ignore_index=True,
)

counts = Counter(
    token
    for text in all_training_texts
    for token in tokenize(text)
)

vocabulary = [
    "<PAD>",
    "<UNK>",
    "<CLS>",
] + sorted(counts)

token2id = {
    token: index
    for index, token
    in enumerate(vocabulary)
}

PAD_ID = token2id["<PAD>"]
UNK_ID = token2id["<UNK>"]
CLS_ID = token2id["<CLS>"]

label_encoder = LabelEncoder()
label_encoder.fit(
    base_frame["label"]
)

print("Vocabulary size:", len(vocabulary))
print("Labels:", list(label_encoder.classes_))

# 14. Dataset and Dynamic Padding

In [ ]:
class TextDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
    ):
        self.frame = frame.reset_index(
            drop=True
        )

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        input_ids = [
            CLS_ID
        ] + [
            token2id.get(
                token,
                UNK_ID,
            )
            for token in tokenize(
                row["text"]
            )
        ]

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long,
            ),
            "label": torch.tensor(
                label_encoder.transform(
                    [row["label"]]
                )[0],
                dtype=torch.long,
            ),
            "text": row["text"],
        }


def collate_batch(batch):
    maximum_length = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = torch.full(
        (
            len(batch),
            maximum_length,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    labels = []

    for row, item in enumerate(batch):
        sequence = item["input_ids"]

        input_ids[
            row,
            :len(sequence),
        ] = sequence

        labels.append(
            item["label"]
        )

    return {
        "input_ids": input_ids,
        "padding_mask": (
            input_ids == PAD_ID
        ),
        "attention_mask": (
            input_ids != PAD_ID
        ).long(),
        "labels": torch.stack(labels),
        "texts": [
            item["text"]
            for item in batch
        ],
    }


def make_loader(
    frame,
    shuffle,
    batch_size=8,
    seed=42,
):
    return DataLoader(
        TextDataset(frame),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_batch,
        generator=(
            torch.Generator()
            .manual_seed(seed)
        ),
    )


base_loader = make_loader(
    base_frame,
    shuffle=True,
)

adaptation_train_loader = make_loader(
    adaptation_train,
    shuffle=True,
)

adaptation_validation_loader = make_loader(
    adaptation_validation,
    shuffle=False,
)

adaptation_test_loader = make_loader(
    adaptation_test,
    shuffle=False,
)

sample_batch = next(
    iter(base_loader)
)

print(
    sample_batch["input_ids"].shape
)

# 15. Self-Attention Block

In [ ]:
class SelfAttention(nn.Module):
    def __init__(
        self,
        model_dimension: int,
        head_count: int,
        dropout: float,
    ):
        super().__init__()

        if (
            model_dimension
            % head_count
            != 0
        ):
            raise ValueError(
                "model_dimension must be divisible by head_count"
            )

        self.model_dimension = model_dimension
        self.head_count = head_count
        self.head_dimension = (
            model_dimension
            // head_count
        )

        self.q_proj = nn.Linear(
            model_dimension,
            model_dimension,
        )
        self.k_proj = nn.Linear(
            model_dimension,
            model_dimension,
        )
        self.v_proj = nn.Linear(
            model_dimension,
            model_dimension,
        )
        self.out_proj = nn.Linear(
            model_dimension,
            model_dimension,
        )
        self.dropout = nn.Dropout(
            dropout
        )

    def _split_heads(
        self,
        tensor: torch.Tensor,
    ) -> torch.Tensor:
        batch_size, sequence_length, _ = (
            tensor.shape
        )

        return (
            tensor.view(
                batch_size,
                sequence_length,
                self.head_count,
                self.head_dimension,
            )
            .transpose(1, 2)
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        query = self._split_heads(
            self.q_proj(
                hidden_states
            )
        )
        key = self._split_heads(
            self.k_proj(
                hidden_states
            )
        )
        value = self._split_heads(
            self.v_proj(
                hidden_states
            )
        )

        scores = (
            query
            @ key.transpose(-2, -1)
            / math.sqrt(
                self.head_dimension
            )
        )

        scores = scores.masked_fill(
            padding_mask[
                :,
                None,
                None,
                :,
            ],
            -1e9,
        )

        weights = torch.softmax(
            scores,
            dim=-1,
        )

        weights = self.dropout(
            weights
        )

        context = weights @ value

        context = (
            context.transpose(1, 2)
            .contiguous()
            .view(
                hidden_states.size(0),
                hidden_states.size(1),
                self.model_dimension,
            )
        )

        return self.out_proj(
            context
        )


class TransformerBlock(nn.Module):
    def __init__(
        self,
        model_dimension: int,
        head_count: int,
        feed_forward_dimension: int,
        dropout: float,
    ):
        super().__init__()

        self.attention = SelfAttention(
            model_dimension,
            head_count,
            dropout,
        )

        self.attention_norm = nn.LayerNorm(
            model_dimension
        )

        self.feed_forward = nn.Sequential(
            nn.Linear(
                model_dimension,
                feed_forward_dimension,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                feed_forward_dimension,
                model_dimension,
            ),
        )

        self.feed_forward_norm = nn.LayerNorm(
            model_dimension
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        hidden_states = (
            self.attention_norm(
                hidden_states
                + self.dropout(
                    self.attention(
                        hidden_states,
                        padding_mask,
                    )
                )
            )
        )

        hidden_states = (
            self.feed_forward_norm(
                hidden_states
                + self.dropout(
                    self.feed_forward(
                        hidden_states
                    )
                )
            )
        )

        return hidden_states

# 16. Tiny Transformer Classifier

In [ ]:
class TinyTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        class_count: int,
        model_dimension: int = 32,
        head_count: int = 4,
        layer_count: int = 2,
        feed_forward_dimension: int = 64,
        maximum_length: int = 64,
        dropout: float = 0.10,
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )

        self.position_embedding = nn.Embedding(
            maximum_length,
            model_dimension,
        )

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    model_dimension,
                    head_count,
                    feed_forward_dimension,
                    dropout,
                )
                for _ in range(
                    layer_count
                )
            ]
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            model_dimension,
            class_count,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        positions = torch.arange(
            input_ids.size(1),
            device=input_ids.device,
        ).unsqueeze(0)

        hidden_states = (
            self.token_embedding(
                input_ids
            )
            + self.position_embedding(
                positions
            )
        )

        for block in self.blocks:
            hidden_states = block(
                hidden_states,
                padding_mask,
            )

        representation = hidden_states[
            :,
            0,
            :,
        ]

        logits = self.classifier(
            self.dropout(
                representation
            )
        )

        return {
            "logits": logits,
            "representation": (
                representation
            ),
        }


DEVICE = torch.device("cpu")

torch.manual_seed(42)

base_model = TinyTransformerClassifier(
    vocabulary_size=len(vocabulary),
    class_count=len(
        label_encoder.classes_
    ),
).to(DEVICE)

print(
    "Total parameters:",
    sum(
        parameter.numel()
        for parameter
        in base_model.parameters()
    ),
)

# 17. Base-Model Pretraining

In [ ]:
loss_function = nn.CrossEntropyLoss()


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def evaluate_model(
    model,
    loader,
):
    model.eval()

    losses = []
    labels_all = []
    predictions_all = []
    probabilities_all = []
    texts_all = []

    with torch.no_grad():
        for batch in loader:
            output = model(
                batch[
                    "input_ids"
                ].to(DEVICE),
                batch[
                    "padding_mask"
                ].to(DEVICE),
            )

            labels = batch[
                "labels"
            ].to(DEVICE)

            loss = loss_function(
                output["logits"],
                labels,
            )

            probabilities = torch.softmax(
                output["logits"],
                dim=1,
            )

            predictions = probabilities.argmax(
                dim=1
            )

            losses.append(
                float(loss.item())
            )
            labels_all.extend(
                labels.cpu().tolist()
            )
            predictions_all.extend(
                predictions.cpu().tolist()
            )
            probabilities_all.extend(
                probabilities.cpu().tolist()
            )
            texts_all.extend(
                batch["texts"]
            )

    return {
        "loss": float(
            np.mean(losses)
        ),
        "accuracy": accuracy_score(
            labels_all,
            predictions_all,
        ),
        "macro_f1": f1_score(
            labels_all,
            predictions_all,
            average="macro",
            zero_division=0,
        ),
        "labels": np.asarray(
            labels_all
        ),
        "predictions": np.asarray(
            predictions_all
        ),
        "probabilities": np.asarray(
            probabilities_all
        ),
        "texts": texts_all,
    }


def train_classifier(
    model,
    train_loader,
    validation_loader=None,
    epochs=35,
    learning_rate=0.003,
    patience=8,
):
    trainable_parameters = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.Adam(
        trainable_parameters,
        lr=learning_rate,
        weight_decay=1e-4,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_metric = float("-inf")
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        losses = []
        norms = []

        for batch in train_loader:
            optimizer.zero_grad()

            output = model(
                batch[
                    "input_ids"
                ].to(DEVICE),
                batch[
                    "padding_mask"
                ].to(DEVICE),
            )

            loss = loss_function(
                output["logits"],
                batch[
                    "labels"
                ].to(DEVICE),
            )

            loss.backward()

            norm = clip_grad_norm_(
                trainable_parameters,
                max_norm=5.0,
            )

            optimizer.step()

            losses.append(
                float(loss.item())
            )
            norms.append(
                float(norm)
            )

        if validation_loader is None:
            metric = -float(
                np.mean(losses)
            )
            validation_loss = np.nan
            validation_f1 = np.nan
        else:
            validation = evaluate_model(
                model,
                validation_loader,
            )
            metric = validation[
                "macro_f1"
            ]
            validation_loss = validation[
                "loss"
            ]
            validation_f1 = validation[
                "macro_f1"
            ]

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(
                    np.mean(losses)
                ),
                "validation_loss": (
                    validation_loss
                ),
                "validation_macro_f1": (
                    validation_f1
                ),
                "gradient_norm": float(
                    np.mean(norms)
                ),
            }
        )

        if (
            metric
            > best_metric
            + 1e-6
        ):
            best_metric = metric
            best_state = copy.deepcopy(
                model.state_dict()
            )
            without_improvement = 0
        else:
            without_improvement += 1

        if (
            validation_loader
            is not None
            and without_improvement
            >= patience
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )


set_seed(42)

pretrained_base_model, base_history = (
    train_classifier(
        base_model,
        base_loader,
        validation_loader=None,
        epochs=30,
        learning_rate=0.004,
    )
)

base_state = copy.deepcopy(
    pretrained_base_model.state_dict()
)

print(
    "Base training loss:",
    round(
        base_history[
            "training_loss"
        ].iloc[-1],
        4,
    ),
)

# 18. LoRA Linear Layer

The base linear layer is frozen. LoRA matrices `A` and `B` remain trainable.
`B` starts at zero, so the adapter initially preserves the base output.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(
        self,
        base_layer: nn.Linear,
        rank: int = 4,
        alpha: float = 8.0,
        dropout: float = 0.05,
    ):
        super().__init__()

        if rank <= 0:
            raise ValueError(
                "rank must be positive"
            )

        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(
            dropout
        )

        for parameter in (
            self.base_layer.parameters()
        ):
            parameter.requires_grad = (
                False
            )

        self.lora_A = nn.Linear(
            base_layer.in_features,
            rank,
            bias=False,
        )

        self.lora_B = nn.Linear(
            rank,
            base_layer.out_features,
            bias=False,
        )

        nn.init.kaiming_uniform_(
            self.lora_A.weight,
            a=math.sqrt(5),
        )

        nn.init.zeros_(
            self.lora_B.weight
        )

    def forward(
        self,
        inputs: torch.Tensor,
    ) -> torch.Tensor:
        base_output = (
            self.base_layer(inputs)
        )

        adapter_output = (
            self.lora_B(
                self.lora_A(
                    self.dropout(inputs)
                )
            )
            * self.scaling
        )

        return (
            base_output
            + adapter_output
        )

    def merged_linear(
        self,
    ) -> nn.Linear:
        merged = nn.Linear(
            self.base_layer.in_features,
            self.base_layer.out_features,
            bias=(
                self.base_layer.bias
                is not None
            ),
        )

        update = (
            self.lora_B.weight
            @ self.lora_A.weight
        ) * self.scaling

        with torch.no_grad():
            merged.weight.copy_(
                self.base_layer.weight
                + update
            )

            if (
                self.base_layer.bias
                is not None
            ):
                merged.bias.copy_(
                    self.base_layer.bias
                )

        return merged

# 19. Adapter Injection

This experiment inserts LoRA into query and value projections and trains the
classification head.

In [ ]:
def freeze_all_parameters(
    model,
):
    for parameter in model.parameters():
        parameter.requires_grad = False


def inject_lora(
    model,
    rank=4,
    alpha=8.0,
    dropout=0.05,
):
    freeze_all_parameters(model)

    for block in model.blocks:
        block.attention.q_proj = (
            LoRALinear(
                block.attention.q_proj,
                rank=rank,
                alpha=alpha,
                dropout=dropout,
            )
        )

        block.attention.v_proj = (
            LoRALinear(
                block.attention.v_proj,
                rank=rank,
                alpha=alpha,
                dropout=dropout,
            )
        )

    for parameter in (
        model.classifier.parameters()
    ):
        parameter.requires_grad = True

    return model


full_model = TinyTransformerClassifier(
    vocabulary_size=len(vocabulary),
    class_count=len(
        label_encoder.classes_
    ),
).to(DEVICE)

full_model.load_state_dict(
    base_state
)

lora_model = TinyTransformerClassifier(
    vocabulary_size=len(vocabulary),
    class_count=len(
        label_encoder.classes_
    ),
).to(DEVICE)

lora_model.load_state_dict(
    base_state
)

lora_model = inject_lora(
    lora_model,
    rank=4,
    alpha=8.0,
    dropout=0.05,
)

# 20. Parameter Counting

In [ ]:
def parameter_summary(
    model,
) -> dict:
    total = sum(
        parameter.numel()
        for parameter
        in model.parameters()
    )

    trainable = sum(
        parameter.numel()
        for parameter
        in model.parameters()
        if parameter.requires_grad
    )

    return {
        "total": total,
        "trainable": trainable,
        "trainable_percent": (
            100.0
            * trainable
            / max(total, 1)
        ),
    }


parameter_comparison = pd.DataFrame(
    [
        {
            "strategy": (
                "Full fine-tuning"
            ),
            **parameter_summary(
                full_model
            ),
        },
        {
            "strategy": "LoRA",
            **parameter_summary(
                lora_model
            ),
        },
    ]
)

parameter_comparison

# 21. Full Fine-Tuning Baseline

In [ ]:
set_seed(42)

(
    trained_full_model,
    full_history,
) = train_classifier(
    full_model,
    adaptation_train_loader,
    adaptation_validation_loader,
    epochs=35,
    learning_rate=0.0015,
    patience=8,
)

print(
    "Full best validation F1:",
    round(
        full_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

# 22. LoRA Fine-Tuning

In [ ]:
set_seed(42)

(
    trained_lora_model,
    lora_history,
) = train_classifier(
    lora_model,
    adaptation_train_loader,
    adaptation_validation_loader,
    epochs=35,
    learning_rate=0.006,
    patience=8,
)

print(
    "LoRA best validation F1:",
    round(
        lora_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

LoRA often uses a larger adapter learning rate than full-model fine-tuning,
but the best value remains task-dependent.

# 23. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    full_history["epoch"],
    full_history[
        "validation_macro_f1"
    ],
    label="Full fine-tuning",
)
plt.plot(
    lora_history["epoch"],
    lora_history[
        "validation_macro_f1"
    ],
    label="LoRA",
)
plt.xlabel("Epoch")
plt.ylabel("Validation macro F1")
plt.title("Full Fine-Tuning Versus LoRA")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    full_history["epoch"],
    full_history[
        "training_loss"
    ],
    label="Full fine-tuning",
)
plt.plot(
    lora_history["epoch"],
    lora_history[
        "training_loss"
    ],
    label="LoRA",
)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Adaptation Training Loss")
plt.legend()
plt.tight_layout()
plt.show()

# 24. Validation Comparison

In [ ]:
full_validation = evaluate_model(
    trained_full_model,
    adaptation_validation_loader,
)

lora_validation = evaluate_model(
    trained_lora_model,
    adaptation_validation_loader,
)

validation_comparison = pd.DataFrame(
    [
        (
            "Full fine-tuning",
            full_validation[
                "accuracy"
            ],
            full_validation[
                "macro_f1"
            ],
        ),
        (
            "LoRA",
            lora_validation[
                "accuracy"
            ],
            lora_validation[
                "macro_f1"
            ],
        ),
    ],
    columns=[
        "Strategy",
        "Validation accuracy",
        "Validation macro F1",
    ],
)

validation_comparison

# 25. Test Evaluation

In [ ]:
full_test = evaluate_model(
    trained_full_model,
    adaptation_test_loader,
)

lora_test = evaluate_model(
    trained_lora_model,
    adaptation_test_loader,
)

test_comparison = pd.DataFrame(
    [
        (
            "Full fine-tuning",
            full_test["accuracy"],
            full_test["macro_f1"],
        ),
        (
            "LoRA",
            lora_test["accuracy"],
            lora_test["macro_f1"],
        ),
    ],
    columns=[
        "Strategy",
        "Test accuracy",
        "Test macro F1",
    ],
)

test_comparison

# 26. Per-Class Metrics

In [ ]:
print("LoRA classification report")
print(
    classification_report(
        label_encoder.inverse_transform(
            lora_test["labels"]
        ),
        label_encoder.inverse_transform(
            lora_test[
                "predictions"
            ]
        ),
        zero_division=0,
    )
)

# 27. Confusion Matrices

In [ ]:
class_names = list(
    label_encoder.classes_
)

lora_matrix = confusion_matrix(
    label_encoder.inverse_transform(
        lora_test["labels"]
    ),
    label_encoder.inverse_transform(
        lora_test["predictions"]
    ),
    labels=class_names,
)

pd.DataFrame(
    lora_matrix,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

# 28. Confidence and Error Analysis

In [ ]:
lora_actual = (
    label_encoder.inverse_transform(
        lora_test["labels"]
    )
)

lora_predicted = (
    label_encoder.inverse_transform(
        lora_test["predictions"]
    )
)

error_frame = pd.DataFrame(
    {
        "text": lora_test["texts"],
        "actual": lora_actual,
        "predicted": lora_predicted,
        "confidence": lora_test[
            "probabilities"
        ].max(axis=1),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame.sort_values(
    ["correct", "confidence"],
    ascending=[True, False],
)

# 29. Adapter-Only State

Adapter-only saving should include LoRA matrices and any explicitly trainable
task head.

In [ ]:
def adapter_state_dict(
    model,
):
    return {
        name: tensor.detach().cpu()
        for name, tensor
        in model.state_dict().items()
        if (
            "lora_A" in name
            or "lora_B" in name
            or name.startswith(
                "classifier."
            )
        )
    }


adapter_state = adapter_state_dict(
    trained_lora_model
)

pd.Series(
    {
        "adapter tensors": len(
            adapter_state
        ),
        "adapter parameters": sum(
            tensor.numel()
            for tensor
            in adapter_state.values()
        ),
    }
)

# 30. Saving and Reloading an Adapter

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    adapter_path = (
        Path(directory)
        / "lora_adapter.pt"
    )

    torch.save(
        {
            "adapter_state_dict": (
                adapter_state
            ),
            "rank": 4,
            "alpha": 8.0,
            "target_modules": [
                "q_proj",
                "v_proj",
            ],
            "base_model": (
                "offline_tiny_transformer"
            ),
        },
        adapter_path,
    )

    loaded = torch.load(
        adapter_path,
        map_location="cpu",
        weights_only=False,
    )

    reloaded_lora_model = (
        TinyTransformerClassifier(
            vocabulary_size=len(
                vocabulary
            ),
            class_count=len(
                label_encoder.classes_
            ),
        ).to(DEVICE)
    )

    reloaded_lora_model.load_state_dict(
        base_state
    )

    reloaded_lora_model = inject_lora(
        reloaded_lora_model,
        rank=loaded["rank"],
        alpha=loaded["alpha"],
        dropout=0.0,
    )

    reloaded_lora_model.load_state_dict(
        loaded[
            "adapter_state_dict"
        ],
        strict=False,
    )

    reload_test = evaluate_model(
        reloaded_lora_model,
        adaptation_test_loader,
    )

print(
    "Reloaded LoRA accuracy:",
    round(
        reload_test["accuracy"],
        3,
    ),
)

# 31. Merging LoRA Weights

In [ ]:
def merge_lora_modules(
    module: nn.Module,
):
    for name, child in list(
        module.named_children()
    ):
        if isinstance(
            child,
            LoRALinear,
        ):
            setattr(
                module,
                name,
                child.merged_linear(),
            )
        else:
            merge_lora_modules(
                child
            )

    return module


merged_model = copy.deepcopy(
    trained_lora_model
)

merged_model = merge_lora_modules(
    merged_model
).to(DEVICE)

print(
    "LoRA modules after merge:",
    sum(
        isinstance(
            module,
            LoRALinear,
        )
        for module
        in merged_model.modules()
    ),
)

# 32. Merge Equivalence Check

In [ ]:
comparison_batch = next(
    iter(
        adaptation_test_loader
    )
)

trained_lora_model.eval()
merged_model.eval()

with torch.no_grad():
    adapter_logits = (
        trained_lora_model(
            comparison_batch[
                "input_ids"
            ].to(DEVICE),
            comparison_batch[
                "padding_mask"
            ].to(DEVICE),
        )["logits"]
    )

    merged_logits = (
        merged_model(
            comparison_batch[
                "input_ids"
            ].to(DEVICE),
            comparison_batch[
                "padding_mask"
            ].to(DEVICE),
        )["logits"]
    )

maximum_difference = float(
    (
        adapter_logits
        - merged_logits
    )
    .abs()
    .max()
    .item()
)

print(
    "Maximum logit difference:",
    maximum_difference,
)

Adapter dropout must be disabled during evaluation for deterministic merge
equivalence.

# 33. Rank Comparison

In [ ]:
rank_accounting = pd.DataFrame(
    [
        (
            rank,
            4
            * lora_parameter_count(
                32,
                32,
                rank,
            ),
        )
        for rank in [
            1,
            2,
            4,
            8,
        ]
    ],
    columns=[
        "Rank",
        "Attention adapter parameters",
    ],
)

rank_accounting

The factor four represents query and value adapters across two blocks in the
offline model.

# 34. Memory and Storage Accounting

In [ ]:
bytes_per_float32 = 4

storage_comparison = pd.DataFrame(
    [
        (
            "Full model",
            parameter_summary(
                trained_full_model
            )["total"],
        ),
        (
            "LoRA adapter and head",
            sum(
                tensor.numel()
                for tensor
                in adapter_state.values()
            ),
        ),
    ],
    columns=[
        "Artifact",
        "Parameter count",
    ],
)

storage_comparison[
    "Approximate float32 KB"
] = (
    storage_comparison[
        "Parameter count"
    ]
    * bytes_per_float32
    / 1024
)

storage_comparison

Optimizer states and gradients can consume more memory than the saved
checkpoint, so parameter count is only one part of training-memory accounting.

# 35. Optional Hugging Face Setup

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec(
        "transformers"
    )
    is not None
)

PEFT_AVAILABLE = (
    importlib.util.find_spec(
        "peft"
    )
    is not None
)

BITSANDBYTES_AVAILABLE = (
    importlib.util.find_spec(
        "bitsandbytes"
    )
    is not None
)

RUN_HUGGING_FACE_DEMOS = False
USE_LOCAL_FILES_ONLY = True

MODEL_ID = (
    "distilbert/"
    "distilbert-base-uncased"
)

pd.Series(
    {
        "transformers installed": (
            TRANSFORMERS_AVAILABLE
        ),
        "peft installed": (
            PEFT_AVAILABLE
        ),
        "bitsandbytes installed": (
            BITSANDBYTES_AVAILABLE
        ),
        "run demos": (
            RUN_HUGGING_FACE_DEMOS
        ),
        "local files only": (
            USE_LOCAL_FILES_ONLY
        ),
        "model ID": MODEL_ID,
    }
)

Installation commands:

```text
python -m pip install transformers peft accelerate
```

Quantized workflows may additionally require a compatible `bitsandbytes`
installation and supported hardware.

# 36. Optional PEFT LoRA Configuration

Target-module names are architecture-specific. DistilBERT commonly exposes
query and value projections with names such as `q_lin` and `v_lin`, but the
model should always be inspected before configuration.

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and PEFT_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from peft import (
        LoraConfig,
        TaskType,
        get_peft_model,
    )
    from transformers import (
        AutoModelForSequenceClassification,
    )

    hf_base_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_ID,
            num_labels=4,
            local_files_only=(
                USE_LOCAL_FILES_ONLY
            ),
        )
    )

    peft_config = LoraConfig(
        task_type=(
            TaskType.SEQ_CLS
        ),
        inference_mode=False,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=[
            "q_lin",
            "v_lin",
        ],
    )

    hf_lora_model = get_peft_model(
        hf_base_model,
        peft_config,
    )

    hf_lora_model.print_trainable_parameters()
else:
    print(
        "Optional PEFT configuration skipped."
    )

# 37. Optional Training Structure

In [ ]:
peft_training_template = '''
from transformers import Trainer, TrainingArguments

arguments = TrainingArguments(
    output_dir="checkpoints/lora-classifier",
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=hf_lora_model,
    args=arguments,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
'''

print(
    peft_training_template
)

Exact Trainer arguments and PEFT integration details should be checked against
the installed library versions.

# 38. Optional Adapter Saving and Loading

In [ ]:
adapter_template = '''
from peft import PeftModel

hf_lora_model.save_pretrained(
    "adapters/domain-classifier"
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=4,
)

loaded_model = PeftModel.from_pretrained(
    base_model,
    "adapters/domain-classifier",
)

merged_model = loaded_model.merge_and_unload()
'''

print(adapter_template)

# 39. Optional Multiple Adapters

Multiple adapters can share one base model when they use compatible adapter
types and configurations. Only the required adapter needs to be activated for
a task.

In [ ]:
multiple_adapter_template = '''
model.load_adapter(
    "adapter_repository_one",
    adapter_name="task_one",
)

model.load_adapter(
    "adapter_repository_two",
    adapter_name="task_two",
)

model.set_adapter("task_one")
'''

print(
    multiple_adapter_template
)

# 40. Optional QLoRA Structure

The following template is not executed. Quantization support depends on the
installed software, operating system, accelerator, and selected data type.

In [ ]:
qlora_template = '''
import torch
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(
    model
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config,
)
'''

print(qlora_template)

# 41. Target-Module Inspection

Never copy target-module names blindly from another architecture.

In [ ]:
def list_linear_module_names(
    model,
) -> list[str]:
    return [
        name
        for name, module
        in model.named_modules()
        if isinstance(
            module,
            nn.Linear,
        )
    ]


list_linear_module_names(
    pretrained_base_model
)[:20]

# 42. Failure Modes

Common failures include:

- wrong target-module names;
- accidentally training the full base model;
- saving only adapter weights without base-model metadata;
- loading an adapter on an incompatible base revision;
- rank too small for the task;
- adapter learning rate too high;
- merging while dropout is active;
- assuming quantization works on unsupported hardware.

In [ ]:
failure_register = pd.DataFrame(
    [
        ("No adapter parameters", "verify target-module names"),
        ("Too many trainable parameters", "inspect requires_grad"),
        ("Poor adaptation", "tune rank, targets, and learning rate"),
        ("Load failure", "match base model and revision"),
        ("Merge mismatch", "use evaluation mode and correct scaling"),
        ("Quantization failure", "check platform and backend support"),
    ],
    columns=["Symptom", "Check"],
)

failure_register

# 43. Arabic and Multilingual Considerations

PEFT can adapt multilingual models without duplicating the full base model for
every language or domain.

Important issues include:

- language balance;
- tokenizer fragmentation;
- MSA versus dialect coverage;
- morphology and attached clitics;
- task interference across adapters;
- fully vocalized versus unvocalized text;
- adapter routing by language or domain.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "fully vocalized MSA",
            "preserve tashkeel",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "clitic plus noun",
            "inspect subword splits",
        ),
        (
            "كِتَابُهُمَا",
            "stem plus pronoun",
            "evaluate morphology",
        ),
    ],
    columns=[
        "Form",
        "Property",
        "Adapter evaluation",
    ],
)

arabic_examples

For fully vocalized Arabic tasks, the base checkpoint, tokenizer, adapter
training data, evaluation data, and inference pipeline should preserve
tashkeel consistently.

A small adapter cannot recover linguistic information that the base tokenizer
or base model systematically discards.

# 44. Reproducibility and Reporting

Report:

- base model ID and exact revision;
- tokenizer ID and revision;
- PEFT method;
- adapter rank, alpha, and dropout;
- target modules;
- modules saved with the adapter;
- trainable and total parameters;
- data splits;
- optimizer and learning rate;
- quantization configuration;
- hardware and precision;
- merge status;
- adapter checkpoint size;
- validation and test metrics;
- random seeds;
- software versions;
- limitations and license.

In [ ]:
reproducibility_metadata = pd.Series(
    {
        "base model": (
            "offline_tiny_transformer"
        ),
        "adapter method": "LoRA",
        "rank": 4,
        "alpha": 8.0,
        "target modules": (
            "q_proj, v_proj"
        ),
        "training examples": len(
            adaptation_train
        ),
        "validation examples": len(
            adaptation_validation
        ),
        "test examples": len(
            adaptation_test
        ),
        "device": str(DEVICE),
        "seed": 42,
        "python": (
            platform.python_version()
        ),
        "torch": torch.__version__,
        "transformers installed": (
            TRANSFORMERS_AVAILABLE
        ),
        "peft installed": (
            PEFT_AVAILABLE
        ),
        "optional demos enabled": (
            RUN_HUGGING_FACE_DEMOS
        ),
    },
    name="Lesson 42 experiment",
)

reproducibility_metadata

# 45. Knowledge Check

1. What problem does PEFT address?
2. How does LoRA differ from full fine-tuning?
3. What do matrices A and B represent?
4. What does LoRA rank control?
5. Why is B commonly initialized to zero?
6. What does alpha divided by rank do?
7. Why must target-module names be inspected?
8. Which parameters remain trainable in the offline LoRA experiment?
9. What should an adapter-only checkpoint contain?
10. How are LoRA weights merged?
11. Why can merged and unmerged outputs differ during training mode?
12. What is QLoRA?
13. Why is parameter count not equal to total training memory?
14. Why must a base revision be recorded?
15. How can PEFT support multilingual deployment?

# 46. Exercises

## Exercise 1 — Rank Sweep

Compare ranks 1, 2, 4, and 8 under identical splits.

## Exercise 2 — Target Modules

Compare query/value adapters with adapters on every attention projection.

## Exercise 3 — Feed-Forward Adapters

Add LoRA to feed-forward linear layers.

## Exercise 4 — Feature-Extraction Baseline

Compare LoRA with a classifier-head-only baseline.

## Exercise 5 — Adapter Storage

Measure full-model and adapter checkpoint sizes on disk.

## Exercise 6 — Adapter Merge

Verify output equivalence before and after merging.

## Exercise 7 — Hugging Face PEFT

Fine-tune a small encoder with `LoraConfig` and `get_peft_model`.

## Exercise 8 — Multiple Domains

Train separate adapters for finance and medical data.

## Exercise 9 — Arabic PEFT

Adapt a multilingual encoder to a fully vocalized MSA classification task.

## Exercise 10 — Model Card

Document the base model, adapter settings, data, limitations, and metrics.

## Challenge Exercises

1. Implement LoRA for convolutional or embedding layers.
2. Compare LoRA with prompt tuning.
3. Add QLoRA on compatible GPU hardware.
4. Evaluate catastrophic interference when merging several adapters.
5. Build an adapter router for multiple languages or domains.

# 47. Summary and Next Lesson

In this lesson:

- PEFT was motivated by training-memory and checkpoint-storage constraints;
- full fine-tuning, feature extraction, adapters, and LoRA were compared;
- LoRA mathematics, rank, scaling, dropout, and target modules were explained;
- a LoRA linear layer was implemented from first principles;
- adapters were injected into query and value attention projections;
- full fine-tuning and LoRA were compared under identical domain-adaptation
  splits;
- trainable-parameter and storage reductions were measured;
- adapter-only saving and loading were implemented;
- LoRA weights were merged and output equivalence was checked;
- Hugging Face PEFT, adapter loading, multiple adapters, and QLoRA workflow
  structures were introduced;
- failure modes, reproducibility, Arabic morphology, multilingual adaptation,
  and tashkeel preservation were integrated.

## Next Lesson

**Lesson 43: Large Language Model Foundations, Scaling, and Instruction
Tuning** begins **Module 8: Large Language Models**, covering model scaling,
pretraining objectives, emergent capabilities, instruction data, supervised
fine-tuning, alignment stages, context windows, and evaluation.

# References

- Hugging Face PEFT documentation: quick tour, LoRA configuration, adapter
  loading, merging, and multiple adapters.
- Hugging Face Transformers documentation: PEFT integration and quantization.
- Hu, E. J. et al. *LoRA: Low-Rank Adaptation of Large Language Models*.
- Dettmers, T. et al. *QLoRA: Efficient Finetuning of Quantized LLMs*.